# Population-Planner EOH Test

Run the new `population_planner` mode on online bin packing from the clean baseline-derived branch, then inspect the planner-facing logs.

In [ ]:
from pathlib import Path
import json
import os
import requests
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
EOH_SRC = PROJECT_ROOT / "eoh" / "src"

if str(EOH_SRC) not in sys.path:
    sys.path.insert(0, str(EOH_SRC))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EOH_SRC:", EOH_SRC)


In [ ]:
# Configure this cell before running.
RUN_NAME = "population_planner_smoke"
OUTPUT_DIR = PROJECT_ROOT / "compare_runs" / RUN_NAME

USE_LOCAL_LLM = False
LOCAL_LLM_URL = "http://127.0.0.1:11012/completions"

# ENSIA HPC vLLM defaults, intentionally hardcoded for this platform.
LLM_API_ENDPOINT = "http://vllm-nodeport.vllm-ns.svc.cluster.local:8000/v1"
LLM_API_KEY = "my-key-ensia-2022-1030"
REQUESTED_LLM_MODEL = "QuantTrio/Qwen3-VL-235B-A22B-Instruct-AWQ"

def resolve_hpc_model(base_url, api_key, requested_model):
    headers = {"Authorization": f"Bearer {api_key}"}
    try:
        resp = requests.get(f"{base_url}/models", headers=headers, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        model_ids = [item.get("id") for item in payload.get("data", []) if item.get("id")]
        print("available models:", model_ids)
        if requested_model in model_ids:
            return requested_model
        if not model_ids:
            print("/models returned no ids; keeping requested model")
            return requested_model
        print(f"requested model '{requested_model}' unavailable, using '{model_ids[0]}'")
        return model_ids[0]
    except Exception as exc:
        print(f"model discovery failed ({exc}); keeping requested model '{requested_model}'")
        return requested_model

LLM_MODEL = resolve_hpc_model(LLM_API_ENDPOINT, LLM_API_KEY, REQUESTED_LLM_MODEL)

POP_SIZE = 4
N_GENERATIONS = 4
N_PROC = 1
EVAL_INSTANCES_PER_GEN = 64
HOLDOUT_INSTANCES = 16
PLANNER_VIEW_SIZE = 8
PROFILE_TRAIN_INSTANCES = 4
PROFILE_HOLDOUT_INSTANCES = 4

DISABLE_NUMBA = False
DEBUG_MODE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
from eoh import eoh
from eoh.utils.getParas import Paras

paras = Paras()
paras.set_paras(
    method="eoh",
    problem="bp_online",
    eoh_mode="population_planner",
    llm_use_local=USE_LOCAL_LLM,
    llm_local_url=LOCAL_LLM_URL,
    llm_api_endpoint=LLM_API_ENDPOINT,
    llm_api_key=LLM_API_KEY,
    llm_model=LLM_MODEL,
    ec_pop_size=POP_SIZE,
    ec_n_pop=N_GENERATIONS,
    exp_n_proc=N_PROC,
    exp_debug_mode=DEBUG_MODE,
    exp_output_path=str(OUTPUT_DIR),
    eval_instances_per_gen=EVAL_INSTANCES_PER_GEN,
    holdout_instances=HOLDOUT_INSTANCES,
    planner_view_size=PLANNER_VIEW_SIZE,
    planner_profile_train_instances=PROFILE_TRAIN_INSTANCES,
    planner_profile_holdout_instances=PROFILE_HOLDOUT_INSTANCES,
)

if DISABLE_NUMBA:
    paras.eva_numba_decorator = False

paras.__dict__


In [ ]:
evolution = eoh.EVOL(paras)
evolution.run()


In [ ]:
RESULTS_DIR = OUTPUT_DIR / "results"
sorted(p.name for p in RESULTS_DIR.iterdir())


In [ ]:
def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

run_rows = read_jsonl(RESULTS_DIR / "run_log.jsonl")
planner_rows = read_jsonl(RESULTS_DIR / "population_planner_output.jsonl")
cards_rows = read_jsonl(RESULTS_DIR / "heuristic_cards.jsonl")
intervention_rows = read_jsonl(RESULTS_DIR / "executed_interventions.jsonl")

print("run rows:", len(run_rows))
print("planner rows:", len(planner_rows))
print("card rows:", len(cards_rows))
print("intervention rows:", len(intervention_rows))


In [ ]:
if run_rows:
    run_rows[-1]


In [ ]:
if planner_rows:
    {
        "planner_view_ids": planner_rows[-1]["planner_view_ids"],
        "summary": planner_rows[-1]["summary"],
        "plan": planner_rows[-1]["plan"],
    }


In [ ]:
if cards_rows:
    latest_cards = cards_rows[-1]["cards"]
    [{
        "id": card["id"],
        "fitness": card["fitness"],
        "diagnosis": card["diagnosis"],
        "behavior": {
            "fragmentation_index": card["behavior"]["fragmentation_index"],
            "resource_opening_rate_early": card["behavior"]["resource_opening_rate_early"],
            "order_sensitivity": card["behavior"]["order_sensitivity"],
            "holdout_gap": card["behavior"]["holdout_gap"],
        },
    } for card in latest_cards[:3]]
